# A6 Keyword & TF-IDF Baselines (CPU, LOCKED TEST)

Melatih baseline keyword + TF-IDF pada train, memilih pada validation, lalu
mengevaluasi locked test **sekali** lewat `sipature_ml.baselines.run_baselines`.
Ikuti `docs/leakage-safe-split-baseline-report.md` sebelum eksekusi.

Input: `data/splits/*` (dari notebook `04`). Output: `artifacts/metrics/`,
`artifacts/models/`, `artifacts/reports/baseline_*`, dan 3 figure.

**PENTING:** notebook ini MEMBACA locked test untuk evaluasi satu-kali. Metric
tidak boleh ditimpa — `run_baselines` menolak jika metric sudah ada. Hasil adalah
*agreement terhadap silver labels*, bukan human-gold accuracy.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

PROJECT_DIR = Path("/content/hackathon/ml")
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
FIGURE_DIR = ARTIFACT_DIR / "figures" / "baselines"

DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"
DRIVE_MODELS_DIR = DRIVE_ROOT / "models"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "baselines"

print("Drive root:", DRIVE_ROOT)
print("Sumber split:", DRIVE_SPLIT_DIR)
print("Artifact dir (lokal):", ARTIFACT_DIR)
print("Figure dir (lokal)  :", FIGURE_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber split: /content/drive/MyDrive/SIPATURE/data/splits
Artifact dir (lokal): /content/hackathon/ml/artifacts
Figure dir (lokal)  : /content/hackathon/ml/artifacts/figures/baselines


In [4]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



In [5]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
4f35162 (HEAD -> main, origin/main, origin/HEAD) feat: add keyword and TF-IDF baselines notebook
7bd5601 docs: record notebook 04 split completion and mark split manifest done
c27d99b Created using Colab


In [6]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 775.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import numpy
import pandas
import pyarrow
import sklearn
import joblib
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("Matplotlib:", matplotlib.__version__)


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
Scikit-learn: 1.7.2
Joblib: 1.5.3
Matplotlib: 3.10.3


In [4]:
# Salin split (train/validation/test + manifest) dari Drive ke lokal.
import shutil
from pathlib import Path

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_files = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "test_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

for filename in split_files:
    source = DRIVE_SPLIT_DIR / filename
    assert source.is_file(), f"Split file tidak ditemukan di Drive: {source}"
    shutil.copy2(source, SPLIT_DIR / filename)
    print("Disalin:", filename)


Disalin: train_silver_v1.jsonl
Disalin: validation_silver_v1.jsonl
Disalin: test_silver_v1.jsonl
Disalin: split_manifest_silver_v1.json


In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


In [6]:
import json
from sipature_ml.config import load_config

config = load_config("training")

manifest = json.loads((SPLIT_DIR / "split_manifest_silver_v1.json").read_text(encoding="utf-8"))
assert manifest.get("test_is_locked"), "Manifest split tidak terkunci (test_is_locked != true)"

print("Experiment version:", config["experiment_version"])
print("Seed:", config["seed"])
print("Task:", config["task"])
print("Split version:", manifest["split_version"])
print("Test is locked:", manifest["test_is_locked"])
print("TF-IDF representations:", config["baselines"]["tfidf"]["representations"])
print("TF-IDF C values:", config["baselines"]["tfidf"]["c_values"])


Experiment version: silver-baselines-1.0.0
Seed: 42
Task: multilabel_aspect_detection
Split version: silver-split-1.0.0
Test is locked: True
TF-IDF representations: ['word', 'char', 'word_char']
TF-IDF C values: [1.0]


In [7]:
# Guard: evaluasi locked test hanya boleh sekali.
import shutil
from pathlib import Path

drive_metrics_guard = DRIVE_METRICS_DIR / "keyword-silver-v1-test-metrics.json"

if drive_metrics_guard.is_file():
    print("Baseline metric SUDAH ada di Drive. Menyalin hasil yang ada, TANPA evaluasi ulang.")
    for drive_dir, local_dir in (
        (DRIVE_METRICS_DIR, ARTIFACT_DIR / "metrics"),
        (DRIVE_MODELS_DIR, ARTIFACT_DIR / "models"),
        (DRIVE_REPORT_DIR, ARTIFACT_DIR / "reports"),
        (DRIVE_FIGURE_DIR, FIGURE_DIR),
    ):
        local_dir.mkdir(parents=True, exist_ok=True)
        for source in sorted(drive_dir.glob("*")):
            if source.is_file():
                shutil.copy2(source, local_dir / source.name)
                print(f"Disalin dari Drive: {source.name} -> {local_dir}")
else:
    from sipature_ml.baselines import run_baselines
    summary = run_baselines(SPLIT_DIR, ARTIFACT_DIR, FIGURE_DIR)
    print("Baseline berhasil dilatih & dievaluasi pada locked test.")
    print("Keyword Macro F1:", round(summary["keyword_test_macro_f1"], 4))
    print("TF-IDF  Macro F1:", round(summary["tfidf_test_macro_f1"], 4))
    print("Keyword Micro F1:", round(summary["keyword_test_micro_f1"], 4))
    print("TF-IDF  Micro F1:", round(summary["tfidf_test_micro_f1"], 4))


Baseline berhasil dilatih & dievaluasi pada locked test.
Keyword Macro F1: 0.9768
TF-IDF  Macro F1: 0.7201
Keyword Micro F1: 0.9783
TF-IDF  Micro F1: 0.804


In [8]:
# Tampilkan metric locked-test (dari file yang baru/telah dibuat).
import json
from pathlib import Path

for name in ("keyword-silver-v1-test-metrics.json", "tfidf-silver-v1-test-metrics.json"):
    path = ARTIFACT_DIR / "metrics" / name
    if not path.is_file():
        print(f"(skip, belum ada) {name}")
        continue
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print(f"=== {name} ===")
    print("  Macro F1:", round(metrics["macro_f1"], 4))
    print("  Micro F1:", round(metrics["micro_f1"], 4))
    print("  Exact Match:", round(metrics["exact_match"], 4))
    print("  Hamming Loss:", round(metrics["hamming_loss"], 4))
    print("  Latency ms/review:", round(metrics["latency_ms_per_review"], 4))


=== keyword-silver-v1-test-metrics.json ===
  Macro F1: 0.9768
  Micro F1: 0.9783
  Exact Match: 0.9455
  Hamming Loss: 0.0039
  Latency ms/review: 1.0687
=== tfidf-silver-v1-test-metrics.json ===
  Macro F1: 0.7201
  Micro F1: 0.804
  Exact Match: 0.7079
  Hamming Loss: 0.0343
  Latency ms/review: 0.3863


In [ ]:
# Salin metrics/models/reports/figure ke Drive (artefak persisten, termasuk subfolder model).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (ARTIFACT_DIR / "metrics", DRIVE_METRICS_DIR),
    (ARTIFACT_DIR / "reports", DRIVE_REPORT_DIR),
    (FIGURE_DIR, DRIVE_FIGURE_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")

# Folder model berisi subfolder (model.joblib + manifest.json) -> copy rekursif.
DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
for sub in sorted((ARTIFACT_DIR / "models").iterdir()):
    if sub.is_dir():
        shutil.copytree(sub, DRIVE_MODELS_DIR / sub.name, dirs_exist_ok=True)
        print(f"Disalin folder: {sub.name} -> {DRIVE_MODELS_DIR}")


In [11]:
# ============================================================
# RUN SUMMARY — hash, metric, dan limitations.
# ============================================================
import json
from pathlib import Path
from sipature_ml.manifest import sha256_file

baseline_summary = json.loads(
    (ARTIFACT_DIR / "reports" / "baseline_summary.json").read_text(encoding="utf-8")
)

print("EXPERIMENT VERSION:", baseline_summary["experiment_version"])
print("SPLIT VERSION      :", baseline_summary["split_version"])
print("TF-IDF representation:", baseline_summary["selected_tfidf_representation"])
print("Keyword test Macro F1:", round(baseline_summary["keyword_test_macro_f1"], 4))
print("TF-IDF  test Macro F1:", round(baseline_summary["tfidf_test_macro_f1"], 4))
print("Figures:", baseline_summary["figures"])

print("\nOUTPUT METRICS DIR :", ARTIFACT_DIR / "metrics")
print("OUTPUT MODELS DIR  :", ARTIFACT_DIR / "models")
print("OUTPUT REPORTS DIR :", ARTIFACT_DIR / "reports")
print("OUTPUT FIGURE DIR  :", FIGURE_DIR)

print("\nREMINDER: metric ini adalah agreement terhadap silver (weak supervision),")
print("bukan akurasi human-gold. Keyword F1 tinggi bersifat circular terhadap silver rules.")


EXPERIMENT VERSION: silver-baselines-1.0.0
SPLIT VERSION      : silver-split-1.0.0
TF-IDF representation: word_char
Keyword test Macro F1: 0.9768
TF-IDF  test Macro F1: 0.7201
Figures: ['34_baseline_silver_test_comparison.png', '35_baseline_per_aspect_f1.png', '36_tfidf_validation_selection.png']

OUTPUT METRICS DIR : /content/hackathon/ml/artifacts/metrics
OUTPUT MODELS DIR  : /content/hackathon/ml/artifacts/models
OUTPUT REPORTS DIR : /content/hackathon/ml/artifacts/reports
OUTPUT FIGURE DIR  : /content/hackathon/ml/artifacts/figures/baselines

REMINDER: metric ini adalah agreement terhadap silver (weak supervision),
bukan akurasi human-gold. Keyword F1 tinggi bersifat circular terhadap silver rules.
